# 3-1. 압축과 캐시

선택 심화. `3`을 마친 뒤 본다.

`3`은 압축으로 입력 토큰을 줄였다. 그런데 비용은 토큰 수가 아니라 단가를 곱한 값이다.
같은 접두부를 다시 보내면 프롬프트 캐시가 걸려 입력 단가가 크게 내려간다. 압축은 지난 메시지를 고치므로 그 접두부를 깬다.
토큰과 캐시 적중과 비용을 함께 재서 압축이 정말 싼지 본다.

### 공통 설정

`3`과 같은 블록이다. 단가에 캐시 입력 칸을 더했다.
캐시 입력 단가는 OpenAI 가격표(openai.com/api/pricing, 2026-10 확인)의 값이고 일반 입력의 10분의 1이다. 가격표가 바뀌면 여기만 고친다.

In [1]:
import json
import os
from pathlib import Path
from dotenv import load_dotenv, find_dotenv
from openai import OpenAI

load_dotenv(find_dotenv(".env", usecwd=True), override=True)
MODEL = os.environ["OPENAI_DEFAULT_MODEL"]
LOWER = os.environ["OPENAI_LOWER_MODEL"]
client = OpenAI()

PRICES = {MODEL: (2.50, 0.25, 15.00), LOWER: (0.75, 0.075, 4.50)}  # 1M 토큰당 (입력, 캐시 입력, 출력) USD
USAGE = []


def keep(resp):
    USAGE.append((resp.model, resp.usage))
    return resp

usage 의 `input_tokens_details.cached_tokens`가 입력 중 캐시에서 읽은 몫이다. 그 몫만 캐시 단가로 계산한다.

In [2]:
def dollars(rows):
    total = 0.0
    for name, u in rows:
        p = next((PRICES[k] for k in sorted(PRICES, key=len, reverse=True) if name.startswith(k)), None)
        if p:
            cached = u.input_tokens_details.cached_tokens
            total += (u.input_tokens - cached) / 1e6 * p[0] + cached / 1e6 * p[1] + u.output_tokens / 1e6 * p[2]
    return total

### 캐시는 접두부가 같을 때만 걸린다

같은 긴 입력을 두 번 보낸다. 두 번째 호출에서 캐시 몫이 생긴다. 캐시는 1024 토큰이 넘는 입력에서만 걸린다.

In [3]:
CORPUS = Path("../../../../../data/corpus/graph")
doc = (CORPUS / "윤봉길.md").read_text(encoding="utf-8")

for label, text in [("첫 호출", doc), ("같은 입력", doc), ("첫 글자만 바꿈", "#" + doc)]:
    u = keep(client.responses.create(model=MODEL, store=False, max_output_tokens=16,
                                     input=text + "\n\n이 글의 제목만 답한다.")).usage
    print(f"{label:10s} 입력 {u.input_tokens:6,d}  캐시 {u.input_tokens_details.cached_tokens:6,d}")

첫 호출       입력  5,247  캐시  4,864
같은 입력      입력  5,247  캐시  4,864
첫 글자만 바꿈   입력  5,247  캐시  4,864


첫 글자 하나만 바뀌어도 뒤가 모두 같은데 캐시 몫이 사라진다. 캐시는 앞에서부터 같은 데까지만 쓴다.
대화로 보면 메시지 배열의 앞부분이 그대로이고 뒤에 붙기만 할 때 가장 잘 걸린다. 캐시 적중은 보장되지 않는다. 같은 입력이어도 가끔 빗나간다.

### 과제와 루프

`3`보다 문서를 늘려 여섯 편을 읽는다. 턴이 길어져야 차이가 보인다.
`parallel_tool_calls=False`로 한 턴에 한 편씩 읽게 한다. 문서가 한 턴에 하나씩 쌓이는 긴 대화를 흉내 낸다.

In [4]:
TARGETS = ["한인애국단", "윤봉길", "이봉창_의거", "강우규", "박열", "백의사"]
TASK = ("다음 여섯 문서를 read_doc 으로 한 번에 하나씩 읽는다: " + ", ".join(TARGETS)
        + ". 다 읽은 뒤 문서마다 핵심 사건을 연도와 함께 한 문장씩 정리한다.")


def read_doc(title):
    return (CORPUS / f"{title}.md").read_text(encoding="utf-8")


READ = [{"type": "function", "name": "read_doc", "description": "문서 한 편의 본문 전체를 돌려준다.", "strict": True,
         "parameters": {"type": "object", "properties": {"title": {"type": "string", "description": "문서 제목"}},
                        "required": ["title"], "additionalProperties": False}}]
print({t: f"{len(read_doc(t)):,}자" for t in TARGETS})

{'한인애국단': '6,032자', '윤봉길': '7,774자', '이봉창_의거': '5,044자', '강우규': '3,396자', '박열': '3,315자', '백의사': '4,153자'}


루프는 `3`과 같다. 트레이스에 턴별 입력과 캐시 몫을 남긴다. `kw`는 API 옵션을 그대로 넘기는 자리다.

In [5]:
def run(task, on_turn=None, max_turns=10, **kw):
    messages = [{"role": "user", "content": task}]
    trace = []
    for turn in range(1, max_turns + 1):
        r = keep(client.responses.create(model=MODEL, input=messages, tools=READ, store=False,
                                         parallel_tool_calls=False, **kw))
        trace.append((r.usage.input_tokens, r.usage.input_tokens_details.cached_tokens))
        calls = [i for i in r.output if i.type == "function_call"]
        if not calls:
            return r.output_text, trace
        messages += [i.model_dump(exclude_none=True) for i in r.output]
        for c in calls:
            messages.append({"type": "function_call_output", "call_id": c.call_id,
                             "output": read_doc(**json.loads(c.arguments))})
        if on_turn:
            messages = on_turn(messages)
    return None, trace

한 방식을 돌리고 턴별 입력과 캐시 몫, 본 호출과 요약 호출의 비용을 나눠 기록한다.
`LOWER` 이름이 `MODEL` 이름으로 시작하므로 요약 호출은 `LOWER`로 먼저 가른다.

In [6]:
RESULTS = {}


def trial(label, **kw):
    mark = len(USAGE)
    answer, trace = run(TASK, **kw)
    rows = USAGE[mark:]
    side = [r for r in rows if r[0].startswith(LOWER)]
    main = [r for r in rows if not r[0].startswith(LOWER)]
    for n, (inp, cached) in enumerate(trace, 1):
        print(f"  턴 {n}  입력 {inp:7,d}  캐시 {cached:7,d}  {cached / inp:4.0%}")
    total_in, total_cached = sum(t[0] for t in trace), sum(t[1] for t in trace)
    RESULTS[label] = {"입력": total_in, "캐시": total_cached, "본 호출": dollars(main),
                      "요약": dollars(side), "합계": dollars(rows)}
    print(f"{label}  입력 합계 {total_in:,}  캐시 {total_cached / total_in:.0%}  ${dollars(rows):.4f}")
    return answer

### 그대로

압축하지 않는다. 입력은 턴마다 문서 한 편씩 자라지만 앞부분은 그대로다.

In [7]:
answer = trial("그대로")
print(answer[:300])

  턴 1  입력     203  캐시       0    0%
  턴 2  입력   4,357  캐시   3,840   88%
  턴 3  입력   9,621  캐시   8,960   93%
  턴 4  입력  12,690  캐시  12,032   95%
  턴 5  입력  14,987  캐시  14,080   94%
  턴 6  입력  17,241  캐시  17,152   99%
  턴 7  입력  20,016  캐시  19,200   96%
그대로  입력 합계 79,115  캐시 95%  $0.0361
여섯 문서를 모두 읽었습니다. 문서마다 핵심 사건을 연도와 함께 한 문장씩 정리하면 다음과 같습니다.

- **한인애국단**: **1931년** 김구가 상하이에서 한인애국단을 결성했고, **1932년** 이봉창 의거와 윤봉길 의거를 실행하며 항일 무장투쟁을 전개했다.  
- **윤봉길**: **1932년 4월 29일** 윤봉길은 상하이 훙커우 공원에서 폭탄을 투척해 일본 고위 인사들을 처단·중상케 한 의거를 감행했다.  
- **이봉창 의거**: **1932년 1월 8일** 이봉창은 도쿄 사쿠라다문에서 일왕 히로히토를 향해 폭탄을 


입력은 계속 늘지만 둘째 턴부터는 대부분이 캐시 몫이다. 새로 붙은 문서 한 편만 정가로 낸다.

### 매 턴 압축

`3`의 `compact`다. 마지막 둘만 원문으로 두고 나머지는 하위 모델 요약으로 갈아 끼운다.
요약 지시만 이 과제에 맞게 바꿨다.

In [8]:
def summarize(body):
    return keep(client.responses.create(
        model=LOWER, store=False,
        input=f"다음 글의 핵심 사건을 연도와 함께 세 문장으로 추린다.\n\n{body[:8000]}")).output_text


def compact(messages, keep_last=2):
    spots = [i for i, m in enumerate(messages) if m.get("type") == "function_call_output"]
    for i in spots[:-keep_last]:
        body = messages[i]["output"]
        if len(body) < 1500 or body.startswith("[압축됨]"):
            continue
        messages[i] = {**messages[i], "output": "[압축됨] " + summarize(body)}
    return messages


answer = trial("매 턴 압축", on_turn=compact)

  턴 1  입력     203  캐시       0    0%
  턴 2  입력   4,357  캐시   3,840   88%
  턴 3  입력   9,621  캐시   8,960   93%
  턴 4  입력   8,704  캐시       0    0%
  턴 5  입력   5,922  캐시       0    0%
  턴 6  입력   5,281  캐시       0    0%
  턴 7  입력   5,952  캐시       0    0%
매 턴 압축  입력 합계 40,040  캐시 32%  $0.0833


압축이 시작된 턴부터 캐시 몫이 거의 사라진다. 매 턴 앞쪽 메시지 하나가 요약으로 바뀌므로 그 뒤가 전부 새 입력이 된다.
입력 합계는 줄었지만 줄어든 토큰이 원래 캐시 단가로 내던 몫이었다. 여기에 요약 호출 값이 더해진다.

### 몰아서 압축

접두부를 덜 깨려면 압축을 드물게, 한꺼번에 한다. 문서 네 편이 쌓일 때마다 마지막 하나만 남기고 모두 요약한다.
압축 직후 한 턴은 캐시를 잃지만 그다음부터 다시 붙는다.

In [9]:
def compact_batch(messages, every=4):
    spots = [i for i, m in enumerate(messages) if m.get("type") == "function_call_output"]
    if len(spots) % every:
        return messages
    for i in spots[:-1]:
        body = messages[i]["output"]
        if len(body) < 1500 or body.startswith("[압축됨]"):
            continue
        messages[i] = {**messages[i], "output": "[압축됨] " + summarize(body)}
    return messages


answer = trial("몰아서 압축", on_turn=compact_batch)

  턴 1  입력     203  캐시       0    0%
  턴 2  입력   4,357  캐시   3,840   88%
  턴 3  입력   9,621  캐시   8,960   93%
  턴 4  입력  12,690  캐시  12,032   95%
  턴 5  입력   3,125  캐시       0    0%
  턴 6  입력   5,379  캐시   2,816   52%
  턴 7  입력   8,154  캐시   4,864   60%
몰아서 압축  입력 합계 43,529  캐시 75%  $0.0461


### 서버 쪽 잘라내기

Responses API 의 `truncation="auto"`는 입력이 컨텍스트 윈도를 넘을 때 서버가 대화 중간 아이템을 버린다. 기본값은 `"disabled"`이고 넘치면 오류를 낸다.
이 과제의 입력은 윈도보다 한참 작으므로 아무것도 버려지지 않는다.

In [10]:
answer = trial("truncation=auto", truncation="auto")

  턴 1  입력     203  캐시       0    0%
  턴 2  입력   4,357  캐시   3,840   88%
  턴 3  입력   9,621  캐시   8,960   93%
  턴 4  입력  12,690  캐시  12,032   95%
  턴 5  입력  14,987  캐시  14,080   94%
  턴 6  입력  17,241  캐시  17,152   99%
  턴 7  입력  20,016  캐시  19,200   96%
truncation=auto  입력 합계 79,115  캐시 95%  $0.0360


입력 합계가 그대로와 같다. 이 옵션은 비용을 줄이는 도구가 아니라 윈도를 넘었을 때 실패 대신 무엇을 할지 정하는 안전장치다.
무엇을 버릴지 고를 수 없으므로 중요한 앞부분이 사라질 수 있다. 캐시 몫의 차이는 실행마다 생기는 적중 편차다.

### 나란히 둔다

답의 품질은 여기서 재지 않는다(`3`과 같은 이유). 같은 질문에 대한 답이 비슷한지는 위에서 눈으로 본다.

In [11]:
base = RESULTS["그대로"]["합계"]
for label, r in RESULTS.items():
    print(f"{label:15s} 입력 {r['입력']:7,d}  캐시 {r['캐시'] / r['입력']:4.0%}"
          f"   본 호출 ${r['본 호출']:.4f}  요약 ${r['요약']:.4f}  합계 ${r['합계']:.4f}  (그대로의 {r['합계'] / base:4.0%})")

그대로             입력  79,115  캐시  95%   본 호출 $0.0361  요약 $0.0000  합계 $0.0361  (그대로의 100%)
매 턴 압축          입력  40,040  캐시  32%   본 호출 $0.0786  요약 $0.0047  합계 $0.0833  (그대로의 231%)
몰아서 압축          입력  43,529  캐시  75%   본 호출 $0.0422  요약 $0.0039  합계 $0.0461  (그대로의 128%)
truncation=auto 입력  79,115  캐시  95%   본 호출 $0.0360  요약 $0.0000  합계 $0.0360  (그대로의 100%)


### 압축을 고르는 기준

캐시가 붙는 대화에서 지난 턴의 토큰은 이미 싸다. 압축은 그 싼 토큰을 줄이는 대신 접두부를 깨고 요약 호출을 더한다.
그래서 이 길이에서는 입력 토큰이 가장 많은 그대로가 가장 싸고, 토큰을 가장 많이 줄인 매 턴 압축이 가장 비싸다. 몰아서 압축은 그 사이다.

압축이 값을 하는 것은 비용이 아니라 윈도 때문일 때다. **대화가 윈도에 가까워지거나, 오래된 결과가 주의를 흐려 답이 나빠질 때 쓴다.**
쓸 때는 드물게, 한꺼번에, 뒤쪽부터 한다. 시스템 지시와 도구 정의처럼 바뀌지 않는 것은 배열 맨 앞에 두고 건드리지 않는다.
`3`의 오프로딩도 같은 눈으로 볼 수 있다. 결과를 처음부터 작게 넣으므로 뒤에 고칠 일이 없어 접두부가 깨지지 않는다.

### 비용 확인

In [ ]:
print(f"호출 {len(USAGE)}회")
print(f"입력 {sum(u.input_tokens for _, u in USAGE):,} 토큰"
      f" (캐시 {sum(u.input_tokens_details.cached_tokens for _, u in USAGE):,})"
      f"   출력 {sum(u.output_tokens for _, u in USAGE):,} 토큰")
print(f"누적 ${dollars(USAGE):.4f}")